# Per-Series SHAP Analysis with ConditionalSHAP

This notebook demonstrates how to use the enhanced `ConditionalSHAP` explainer for:

1. **Batch SHAP computation** - Fast computation for tree-based models (LightGBM)
2. **Per-series analysis** - Compute SHAP values separately for each series
3. **Hierarchical aggregation** - Aggregate importance across hierarchy levels
4. **Visualizations** - Beeswarm/violin plots per series and cohort

The `ConditionalSHAP` class automatically detects the model type and uses:
- **TreeExplainer** for tree-based models (LightGBM, XGBoost, RandomForest) - fast batch computation
- **KernelExplainer** for other models - series-specific background data

**Requirements:**
```bash
pip install xeries[skforecast] lightgbm
```

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

## 1. Create Hierarchical Multi-Series Data

We'll create a dataset with a simple hierarchy:
- **State** (top level): TX, WI
- **Store** (bottom level): multiple stores per state

Series IDs follow the pattern: `{STATE}_{STORE}` (e.g., TX_S1, WI_S2)

In [ ]:
def create_hierarchical_series(n_periods: int = 500) -> dict[str, pd.Series]:
    """Create synthetic hierarchical time series data.
    
    Returns dict of series for skforecast (recommended format).
    """
    # Hierarchy: State -> Store
    hierarchy = {
        "TX": ["TX_S1", "TX_S2", "TX_S3"],
        "WI": ["WI_S1", "WI_S2"],
    }
    
    dates = pd.date_range("2022-01-01", periods=n_periods, freq="h")
    series_dict = {}
    
    for state, stores in hierarchy.items():
        state_effect = 10 if state == "TX" else -5
        
        for store in stores:
            base = 100 + np.random.randn() * 20
            trend = np.linspace(0, 10, n_periods)
            seasonal = 15 * np.sin(2 * np.pi * np.arange(n_periods) / 24)  # Daily pattern
            weekly = 10 * np.sin(2 * np.pi * np.arange(n_periods) / 168)   # Weekly pattern
            noise = np.random.randn(n_periods) * 3
            
            values = base + trend + seasonal + weekly + state_effect + noise
            series_dict[store] = pd.Series(values, index=dates, name=store)
    
    return series_dict

# Create series data
series_dict = create_hierarchical_series(n_periods=500)

print(f"Number of series: {len(series_dict)}")
print(f"Series IDs: {list(series_dict.keys())}")
print(f"Points per series: {len(list(series_dict.values())[0])}")

## 2. Train Forecaster with LightGBM

Using skforecast's `ForecasterRecursiveMultiSeries` with LightGBM and StandardScaler.

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.preprocessing import StandardScaler
from skforecast.recursive import ForecasterRecursiveMultiSeries

# Create and fit forecaster
forecaster = ForecasterRecursiveMultiSeries(
    regressor=LGBMRegressor(
        n_estimators=100,
        max_depth=8,
        learning_rate=0.1,
        random_state=42,
        verbose=-1,
    ),
    lags=24,  # 24 hourly lags
    transformer_series=StandardScaler(),
)

forecaster.fit(series=series_dict)

print(f"Forecaster fitted with {forecaster.regressor.n_features_in_} features")
print(f"Series: {forecaster.series_names_in_}")

## 3. Create Adapter and Extract Training Data

In [ ]:
from xeries.adapters import from_skforecast

# Create adapter (same series as fit)
adapter = from_skforecast(forecaster, series=series_dict)

# Get training data
X, y = adapter.get_training_data()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nFeature names: {adapter.get_feature_names()}")
print(f"\nSeries IDs: {adapter.get_series_ids()}")
print(f"\nX columns: {list(X.columns)}")

In [ ]:
# Show sample of data
X.head()

## 4. ConditionalSHAP with Auto-Detection

The `ConditionalSHAP` explainer automatically detects that we're using a LightGBM model and uses `TreeExplainer` for fast batch computation.

In [ ]:
from xeries import ConditionalSHAP

# Get feature names (numeric columns only, excluding series identifier)
feature_names = adapter.get_feature_names()

# Create explainer - auto-detects TreeExplainer for LightGBM
explainer = ConditionalSHAP(
    model=adapter,
    background_data=X,
    series_col="_level_skforecast",  # skforecast's series column
)

print(f"Batch capable (fast): {explainer._is_batch_capable}")
print(f"Feature names: {feature_names}")

## 5. Full Dataset SHAP Computation

Compute SHAP values for the entire dataset in one batch (fast for tree models).

In [ ]:
# Compute SHAP values for full dataset
result = explainer.explain(X, feature_names=feature_names)

print(f"SHAP values shape: {result.shap_values.shape}")
print(f"Features: {result.feature_names}")
print(f"Series tracked: {result.series_ids.unique().tolist()}")

In [ ]:
# Global importance
result.mean_abs_shap()

In [ ]:
# Convert to DataFrame for easy inspection
shap_df = result.to_dataframe()
print(f"SHAP DataFrame shape: {shap_df.shape}")
shap_df.head()

## 6. Per-Series SHAP Analysis

Use `explain_per_series()` to compute SHAP values separately for each store.

In [ ]:
# Compute SHAP values per series
per_series_results = explainer.explain_per_series(
    X, 
    series_col="_level_skforecast",
    feature_names=feature_names
)

print(f"Series analyzed: {list(per_series_results.keys())}")

# Show top 5 features for each store
for store_id, store_result in per_series_results.items():
    print(f"\n{store_id}:")
    print(store_result.mean_abs_shap().head(5).to_string(index=False))

In [ ]:
# Aggregate importance by series
importance_by_series = result.mean_abs_shap_by_series()
importance_by_series

## 7. Visualize Per-Series Importance

In [ ]:
n_series = len(per_series_results)
fig, axes = plt.subplots(1, n_series, figsize=(4*n_series, 5), sharey=True)

for ax, (store_id, store_result) in zip(axes, per_series_results.items()):
    importance = store_result.mean_abs_shap().head(10)
    ax.barh(importance["feature"], importance["mean_abs_shap"])
    ax.set_title(f"Store: {store_id}")
    ax.invert_yaxis()

axes[0].set_ylabel("Feature")
fig.suptitle("Top 10 Features by Store", fontsize=14)
plt.tight_layout()
plt.show()

## 8. Hierarchical Aggregation

Use `HierarchyDefinition` and `HierarchicalExplainer` to aggregate SHAP values at different levels.

We'll use a parse pattern to extract state from series IDs like `TX_S1` -> state=`TX`.

In [ ]:
from xeries.hierarchy import HierarchyDefinition, HierarchicalExplainer

# Define hierarchy using parse pattern
# Series IDs are like: TX_S1, TX_S2, WI_S1, etc.
# Pattern extracts state (TX, WI) and store (S1, S2, etc.)
hierarchy = HierarchyDefinition(
    levels=["state", "store"],
    parse_pattern=r"(?P<state>\w{2})_(?P<store>\w+)",
    series_col="_level_skforecast"
)

# Verify the hierarchy works
print("Cohorts at each level:")
for level in hierarchy.levels:
    cohorts = hierarchy.get_cohorts(X, level)
    print(f"  {level}: {list(cohorts.keys())}")

In [ ]:
# Create hierarchical explainer
hierarchical = HierarchicalExplainer(explainer, hierarchy)

# Compute hierarchical results (include_raw=True for violin plots)
hier_result = hierarchical.explain(X, include_raw=True)

print(f"Levels: {hier_result.levels}")
print(f"Features: {hier_result.features}")

In [ ]:
# Global importance
print("Global Importance (top 10):")
global_df = hier_result.get_level_df("global")
global_df

In [ ]:
# State-level importance
print("State-Level Importance:")
hier_result.get_level_df("state")

In [ ]:
# Store-level importance
print("Store-Level Importance:")
hier_result.get_level_df("store")

## 9. Hierarchy Visualizations

In [ ]:
from xeries.visualization import plot_hierarchy_summary, plot_hierarchy_bar, plot_hierarchy_violin

# Summary view - grid of all levels
fig, axes = plot_hierarchy_summary(hier_result, top_n=10)
plt.suptitle("Feature Importance Across Hierarchy", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# State comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

plot_hierarchy_bar(hier_result, level="state", cohort="TX", ax=axes[0], top_n=10)
plot_hierarchy_bar(hier_result, level="state", cohort="WI", ax=axes[1], top_n=10)

plt.tight_layout()
plt.show()

In [ ]:
# Violin/beeswarm plot for TX state
fig, ax = plot_hierarchy_violin(hier_result, level="state", cohort="TX", top_n=10)
plt.tight_layout()
plt.show()

## 10. Compare Feature Rankings Across Cohorts

In [ ]:
# Compare cohorts at store level
comparison = hierarchical.compare_cohorts(X, level="store", top_n=10)
print("Feature Importance Comparison (stores):")
comparison

In [ ]:
# Feature ranking stability across stores
stability = hierarchical.feature_ranking_stability(X, level="store", top_n=10)
print("Feature Ranking Stability:")
stability

In [ ]:
# Heatmap of importance across stores
from xeries.visualization import plot_hierarchy_heatmap

fig, ax = plot_hierarchy_heatmap(hier_result, level="store", top_n=15)
plt.tight_layout()
plt.show()

## 11. Using KernelExplainer with Series-Specific Background

For non-tree models or when you want series-specific baselines, use `explainer_type='kernel'`.

**Note:** KernelExplainer is much slower than TreeExplainer, so we'll only compute for a small subset.

In [ ]:
# Force KernelExplainer with series-specific background
kernel_explainer = ConditionalSHAP(
    model=adapter,
    background_data=X,
    series_col="_level_skforecast",
    explainer_type="kernel",
    background_strategy="series",  # Use series-specific background
    n_background_samples=30,
)

print(f"Batch capable: {kernel_explainer._is_batch_capable}")
print(f"Series backgrounds prepared: {len(kernel_explainer._series_backgrounds)}")

In [ ]:
# Compute SHAP for a small subset (KernelExplainer is slow)
X_small = X.groupby("_level_skforecast").head(3)  # 3 samples per series
print(f"Computing SHAP for {len(X_small)} samples...")

kernel_result = kernel_explainer.explain(X_small, feature_names=feature_names)

print(f"\nSHAP values shape: {kernel_result.shap_values.shape}")
print("\nGlobal importance (KernelExplainer):")
kernel_result.mean_abs_shap().head(10)

## Summary

The enhanced `ConditionalSHAP` provides:

1. **Auto-detection** of model type for optimal explainer selection (TreeExplainer for LightGBM)
2. **Batch computation** for tree models (fast)
3. **Series-specific background** for kernel explainer (when needed)
4. **`explain_per_series()`** for detailed per-series analysis
5. **Integration with `HierarchicalExplainer`** for multi-level aggregation
6. **Full evaluation support** - entire dataset, no sampling required
7. **Visualization** at each hierarchy level (beeswarm, bar, heatmap)